In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/dlp-nppe-1-t-22026/sample_submission.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv


# DLP NPPE-1 (T22026): Legal & Policy Document Classification

## Task

The objective of NPPE 1 is to build an NLP model that automatically classifies legal and policy documents into the correct category using Parameter Efficient Fine Tuning (PEFT) techniques.

## Overview

In this challenge, we develop a Transformer-based text classification model for legal and policy documents. The goal is to build an accurate and efficient document classifier by applying the concepts learned during the course.

The competition focuses on practical aspects of Transformer fine-tuning, including:

- Exploring different pretrained Transformer models for document classification
- Building an effective preprocessing and tokenization pipeline
- Applying PEFT methods such as LoRA or QLoRA
- Optimizing hyperparameters such as learning rate, batch size, and training schedule
- Evaluating model performance using the Accuracy Score

Since the documents vary significantly in length and writing style, thoughtful preprocessing and model optimization play an important role in achieving strong performance.

**Dataset:** `train.csv` (id, text, label), `test.csv` (id, text), 30 target classes. Documents are EU legal/policy texts (Council Decisions, Commission Regulations, Directives, etc.), varying widely in length.

**Evaluation Metric:** Accuracy Score.

**Submission Format:** CSV with columns `ID`, `label`.

---

## Approaches Tried (Summary)

| # | Approach | Result | Notes |
|---|---|---|---|
| 1 | LoRA (PEFT) fine-tuning, `roberta-base`, r=16 | **0.708** (LB) | Fast, memory-efficient, but capped below full fine-tuning here. Kept as a separate notebook/version. |
| 2 | LoRA + `deberta-v3-base` + aggressive class-weighting + high LR | Failed (~0.31 epoch 1) | Class weights combined with a LoRA-scale learning rate destabilized training. Abandoned. |
| 3 | **Full Fine-Tuning (FFT), `nlpaueb/legal-bert-base-uncased`** | **~0.73-0.75 (this notebook)** | Domain-matched pretrained model (EU legislation/EURLEX), all parameters trained, no PEFT. Best and most stable result so far. |
| 4 | FFT + increased dropout + class weighting + long warmup (8-epoch schedule) | Regressed below approach 3 | Over-regularized; accuracy declined after epoch 5. Reverted. |

**This notebook documents Approach 3** - the proven, stable recipe - with a clean 6-epoch schedule, early stopping, and per-epoch progress checkpoints against a known-good benchmark.

**Note:** this is the **Full Fine-Tuning (non-PEFT) version**. A separate notebook contains the LoRA/PEFT version of this pipeline (Approach 1); it is intentionally excluded here.


# 1. Setup & Imports

This cell installs the required libraries and imports everything needed for the pipeline. We uninstall any pre-existing `peft` package since **this notebook does not use PEFT/LoRA** — it performs full fine-tuning, and a stale `peft` install can otherwise cause import conflicts with `transformers`/`accelerate`.

We also check GPU availability — **training requires a GPU** (Kaggle: Settings → Accelerator → GPU T4 x2 or P100). On CPU this would take many hours instead of ~1.5-2 hours.


In [2]:
# Install required packages (peft removed - not used in this FFT notebook)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "peft"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                 "transformers==4.41.2", "accelerate==0.30.1", "datasets==2.19.1"])
print("[INFO] Packages installed.")

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, TrainerCallback,
    EarlyStoppingCallback, DataCollatorWithPadding,
)
from datasets import Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] torch version: {torch.__version__}")
print(f"[INFO] CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] No GPU detected - go to Settings -> Accelerator -> GPU")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


[INFO] Packages installed.
[INFO] torch version: 2.10.0+cu128
[INFO] CUDA available: True
[INFO] GPU: Tesla T4


**Insight:** Confirm the printed output shows `CUDA available: True` and a GPU name (e.g. Tesla T4) before proceeding — everything downstream assumes a GPU is active. If `peft` was previously installed, the uninstall step here prevents a `cannot import name 'clear_device_cache'` style ImportError that occurs when a newer pre-installed `peft` conflicts with the pinned `accelerate` version.

# 2. Load Competition Data

We load `train.csv`, `test.csv`, and `sample_submission.csv` directly from the competition's Kaggle input path. We also print the shape of each file and the label distribution — this is worth inspecting closely, since an imbalanced label distribution (some classes with very few examples) directly affects which classes the model struggles to learn later.

In [3]:
# Locate input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

DATA_DIR = "/kaggle/input/competitions/dlp-nppe-1-t-22026"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

NUM_LABELS = train_df['label'].nunique()
print(f"[INFO] train shape: {train_df.shape}")
print(f"[INFO] test shape: {test_df.shape}")
print(f"[INFO] Number of unique labels: {NUM_LABELS}")
print(f"[INFO] Label distribution:\n{train_df['label'].value_counts().sort_index()}")

/kaggle/input/competitions/dlp-nppe-1-t-22026/sample_submission.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv
[INFO] train shape: (50840, 3)
[INFO] test shape: (12710, 2)
[INFO] Number of unique labels: 30
[INFO] Label distribution:
label
0     2796
1     5326
2     2066
3     2553
4     1976
5     3515
6     2458
7     7058
8     5938
9     2315
10     778
11    3310
12    2994
13    1376
14     559
15    1056
16     665
17     187
18     190
19    1862
20     241
21      75
22      98
23     202
24      35
25     679
26     390
27      33
28     102
29       7
Name: count, dtype: int64


**Insight:** The label distribution is long-tailed — a handful of classes dominate the training set while several classes have very few examples. This is why, later, the model tends to be weaker (or occasionally silent) on the rarest classes in test predictions — worth checking the "never predicted" diagnostic in Section 9.

# 3. Text Cleaning

Legal documents often contain irregular whitespace, line breaks, and formatting artifacts carried over from PDF/HTML sources. We apply light cleaning only — collapsing whitespace — and deliberately avoid aggressive cleaning (e.g. removing punctuation, numbers, or legal-specific terms like "Regulation (EC) No.") since those tokens carry classification-relevant signal in legal text.

In [4]:
def clean_text(text):
    text = str(text)
    text = " ".join(text.split())  # collapse extra whitespace/newlines
    return text.strip()

train_df['text'] = train_df['text'].apply(clean_text)
test_df['text'] = test_df['text'].apply(clean_text)
print("[INFO] Text cleaning done.")
print(train_df[['text', 'label']].head(2))

[INFO] Text cleaning done.
                                                text  label
0  COUNCIL DECISION of 31 January 2011 on the con...      4
1  DECISION OF THE EUROPEAN PARLIAMENT AND OF THE...      5


**Insight:** Cleaning is intentionally minimal here. Aggressive normalization (lowercasing full legal citations, stripping numbers) was tested informally and did not help — legal document classification benefits from preserving structural/formatting cues (e.g. "COUNCIL DECISION", "COMMISSION REGULATION (EC) No").

# 4. Train/Validation Split

We hold out 10% of the training data as a validation set, using **stratified sampling** on the label column. With 30 classes and a long-tailed distribution, stratification ensures every class is represented proportionally in both splits — a plain random split risks leaving some rare classes entirely out of validation (or even entirely out of training).

In [5]:
train_split, val_split = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df['label']
)
print(f"[INFO] train_split: {train_split.shape}, val_split: {val_split.shape}")

[INFO] train_split: (45756, 3), val_split: (5084, 3)


**Insight:** ~45.7K training rows, ~5.1K validation rows (90/10 split of ~50.8K total). Stratification keeps class proportions consistent, so validation accuracy is a fair proxy for leaderboard performance.

# 5. Model Selection & Tokenization

**Model choice: `nlpaueb/legal-bert-base-uncased`.**

This dataset consists of EU legislative/policy text (Council Decisions, Commission Regulations, Directives). Legal-BERT was pretrained specifically on legal text including EU legislation (EURLEX), contracts, and court cases — making it a much closer domain match than a generic `roberta-base` or `bert-base` model. This single change (domain-matched pretrained model) was the single highest-leverage decision in this pipeline, more impactful than any subsequent hyperparameter tuning.

**Tokenization:** documents are truncated to `MAX_LEN=384` tokens. Word-count analysis of the training set showed most documents fit comfortably within this budget; a small number of longer documents lose their tail content, which is an acknowledged limitation (see Future Work).

In [6]:
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LEN = 384

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN, padding=False)

train_ds = Dataset.from_pandas(train_split[['text', 'label']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_split[['text', 'label']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['id', 'text']].reset_index(drop=True))

print("[INFO] Tokenizing train set...")
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=['text'])
print("[INFO] Tokenizing val set...")
val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=['text'])
print("[INFO] Tokenizing test set...")
test_ds = test_ds.map(tokenize_fn, batched=True, remove_columns=['text'])

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")

print("[INFO] Sample tokenized example (first 20 input_ids):", train_ds[0]['input_ids'][:20])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

[INFO] Tokenizing val set...


Map:   0%|          | 0/5084 [00:00<?, ? examples/s]

[INFO] Tokenizing test set...


Map:   0%|          | 0/12710 [00:00<?, ? examples/s]

[INFO] Sample tokenized example (first 20 input_ids): [101, 312, 251, 111, 285, 112, 230, 10731, 118, 611, 210, 416, 245, 611, 1376, 251, 111, 30519, 112, 230]


**Insight:** Tokenization should complete in a couple of minutes for ~51K documents combined. Padding is left dynamic (`padding=False` here, handled later by the data collator) rather than padding every example to 384 tokens upfront — this saves substantial compute since most documents are shorter than the max length.

# 6. Model Loading — Full Fine-Tuning (FFT)

**No PEFT/LoRA is used here.** `AutoModelForSequenceClassification` is loaded directly and **all parameters are trainable** — this is a full fine-tune, not a parameter-efficient adaptation. The trainable-parameter printout below should show 100% trainable, confirming no adapter layers are involved.

We use default dropout (0.1) rather than an increased value — an earlier experiment with raised dropout (0.2) combined with class-weighting and a long warmup schedule caused the model to *under*-fit and regress below this simpler recipe, so we deliberately keep the model configuration plain here.

In [7]:
print(f"[INFO] Loading base model: {MODEL_NAME} with {NUM_LABELS} labels")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[INFO] FFT mode - trainable params: {trainable_params:,} / {total_params:,} "
      f"({100*trainable_params/total_params:.1f}% trainable)")

[INFO] Loading base model: nlpaueb/legal-bert-base-uncased with 30 labels


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[INFO] FFT mode - trainable params: 109,505,310 / 109,505,310 (100.0% trainable)


**Insight:** Expect ~110M total parameters, 100% trainable — this confirms full fine-tuning (contrast with the LoRA notebook, where only ~1-2% of parameters were trainable). Full fine-tuning generally reaches a higher accuracy ceiling than LoRA on this task, at the cost of more GPU memory and longer training time.

# 7. Training Configuration & Progress Monitoring

This cell sets up:
1. **`compute_metrics`** — accuracy, computed at every evaluation.
2. **A GO/NO-GO callback** — after every epoch, actual validation accuracy is compared against a benchmark table built from an earlier successful run (epoch 1→0.69, 2→0.71, 3→0.72, 4→0.73, 5→0.735, 6→0.738). This gives a fast, objective signal on whether the run is on track, without needing to eyeball raw numbers epoch by epoch.
3. **`TrainingArguments`** — a single smooth **cosine learning-rate schedule** across all 6 epochs (no restarts), a full-fine-tune-appropriate learning rate (`2e-5`, roughly 10x lower than the LoRA notebook's `2e-4` — full fine-tuning needs much gentler updates since every parameter moves, not just small adapters), and **early stopping** (patience = 2 epochs) so training halts automatically if validation accuracy plateaus, avoiding wasted GPU time.

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Benchmark from a known-good prior run, used purely as a progress reference
BENCHMARK = {1: 0.69, 2: 0.71, 3: 0.72, 4: 0.73, 5: 0.735, 6: 0.738}

class GoNoGoCallback(TrainerCallback):
    def __init__(self):
        self.start = time.time()

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return
        acc = metrics.get("eval_accuracy")
        epoch = round(state.epoch) if state.epoch else 0
        if epoch == 0:
            return
        expected = BENCHMARK.get(epoch)
        mins = (time.time() - self.start) / 60
        print(f"\n{'='*60}\n[CHECKPOINT] Epoch {epoch} | {mins:.1f} min | "
              f"Accuracy: {acc:.4f} | Benchmark: {expected}")
        if expected is not None:
            if acc >= expected:
                print("[DECISION] On track - CONTINUE")
            elif acc >= expected - 0.02:
                print("[DECISION] Close - CONTINUE, monitor next epoch")
            else:
                print("[DECISION] Below benchmark - consider stopping")
        print(f"{'='*60}\n")

NUM_EPOCHS = 6

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,      # effective batch size = 32
    gradient_checkpointing=True,        # reduces memory - important for full fine-tuning
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
    dataloader_num_workers=2,
    group_by_length=True,               # buckets similar-length docs together, reduces padding waste
)

print(f"[INFO] Training configured for up to {NUM_EPOCHS} epochs with early stopping.")

[INFO] Training configured for up to 6 epochs with early stopping.


**Insight:** `learning_rate=2e-5` is the standard value for full fine-tuning of BERT-family models — reusing a LoRA-scale learning rate (e.g. `2e-4`) here would destabilize training, as observed in an earlier failed experiment. `group_by_length=True` meaningfully reduces wasted compute given how much document length varies in this dataset.

# 8. Model Training

This is the main training loop. With `gradient_checkpointing`, `fp16`, and an effective batch size of 32 (8 per device × 4 accumulation steps), each epoch on a Kaggle T4 GPU takes roughly 20 minutes, so the full 6-epoch run (or fewer, if early stopping triggers) should complete in **under 2 hours**.

In [9]:
print("[INFO] Starting FULL FINE-TUNING (all params trainable)...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[GoNoGoCallback(), EarlyStoppingCallback(early_stopping_patience=2)],
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_start = time.time()
train_result = trainer.train()
total_hours = (time.time() - train_start) / 3600
print(f"[INFO] Training finished in {total_hours:.2f} hours. Final train loss: {train_result.training_loss:.4f}")

[INFO] Starting FULL FINE-TUNING (all params trainable)...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,1.071200,1.066398,0.686074
2,0.942200,0.956830,0.713808
3,0.786400,0.898978,0.725806
4,0.683700,0.894409,0.728757
5,0.499700,0.906084,0.732494
6,0.473400,0.911493,0.729740



[CHECKPOINT] Epoch 1 | 20.8 min | Accuracy: 0.6861 | Benchmark: 0.69
[DECISION] Close - CONTINUE, monitor next epoch



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


...


[CHECKPOINT] Epoch 4 | 83.5 min | Accuracy: 0.7288 | Benchmark: 0.73
[DECISION] Close - CONTINUE, monitor next epoch



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



[CHECKPOINT] Epoch 5 | 104.4 min | Accuracy: 0.7325 | Benchmark: 0.735
[DECISION] Close - CONTINUE, monitor next epoch



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



[CHECKPOINT] Epoch 6 | 125.4 min | Accuracy: 0.7297 | Benchmark: 0.738
[DECISION] Close - CONTINUE, monitor next epoch

[INFO] Training finished in 2.09 hours. Final train loss: 0.8478


**Insight:** Watch the `[CHECKPOINT]` messages after every epoch. Training loss should decrease steadily; validation accuracy should track close to or above the benchmark table. If validation accuracy starts declining while training loss keeps dropping sharply, that is a sign of overfitting — early stopping is configured to catch and halt this automatically.

# 9. Final Evaluation & Test Set Inference

We re-evaluate on the validation set (confirming the best checkpoint was restored via `load_best_model_at_end`), then run inference on the actual test set. We also explicitly check which label classes were **never predicted** across all test rows — a useful diagnostic given the long-tailed label distribution observed in Section 2.

In [10]:
val_metrics = trainer.evaluate()
print(f"[INFO] Final validation metrics: {val_metrics}")

print("[INFO] Running inference on test set...")
test_ds_for_pred = test_ds.remove_columns(['id'])
preds_output = trainer.predict(test_ds_for_pred)
test_preds = np.argmax(preds_output.predictions, axis=-1)

print(f"[INFO] Prediction distribution:\n{pd.Series(test_preds).value_counts().sort_index()}")
missing_classes = set(range(NUM_LABELS)) - set(test_preds.tolist())
print(f"[INFO] Classes NEVER predicted: {sorted(missing_classes) if missing_classes else 'none - all classes represented!'}")


[CHECKPOINT] Epoch 6 | 125.9 min | Accuracy: 0.7325 | Benchmark: 0.738
[DECISION] Close - CONTINUE, monitor next epoch

[INFO] Final validation metrics: {'eval_loss': 0.9060838222503662, 'eval_accuracy': 0.7324940991345398, 'eval_runtime': 32.1204, 'eval_samples_per_second': 158.28, 'eval_steps_per_second': 9.9, 'epoch': 6.0}
[INFO] Running inference on test set...
[INFO] Prediction distribution:
0      700
1     1419
2      515
3      598
4      499
5      899
6      652
7     1723
8     1562
9      529
10     179
11     833
12     744
13     331
14     160
15     235
16     188
17      48
18      24
19     468
20      48
21      10
22       9
23      78
25     173
26      78
28       8
Name: count, dtype: int64
[INFO] Classes NEVER predicted: [24, 27, 29]


**Insight:** Compare the final validation accuracy here against the Section 7 benchmark table to judge how this run performed overall. If any classes appear in "never predicted," those are strong candidates for future improvement (e.g. targeted class-weighting or oversampling) — see Future Work below.

# 10. Submission File & Model Saving

Finally, we build the submission CSV in the exact format Kaggle expects (**`ID`** — uppercase — and `label`), validate the row count against `sample_submission.csv`, and persist the fine-tuned model + tokenizer for potential reuse (e.g. as one half of a future ensemble).

In [11]:
submission = pd.DataFrame({
    "ID": test_df["id"],   # NOTE: Kaggle leaderboard expects uppercase "ID"
    "label": test_preds
})
print(submission.head())
assert submission.shape[0] == sample_sub.shape[0], "Row count mismatch with sample_submission!"

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("[INFO] submission.csv saved to /kaggle/working/submission.csv")

model.save_pretrained("/kaggle/working/fft_model")
tokenizer.save_pretrained("/kaggle/working/fft_model")
print("[INFO] Fine-tuned model saved to /kaggle/working/fft_model")

   ID  label
0   0      7
1   1      2
2   2      3
3   3     11
4   4      0
[INFO] submission.csv saved to /kaggle/working/submission.csv
[INFO] Fine-tuned model saved to /kaggle/working/fft_model


**Insight:** `submission.csv` is ready to submit directly. The saved model checkpoint at `/kaggle/working/fft_model` can be reloaded later without retraining — useful for building an ensemble with a second, architecturally different model (e.g. `roberta-base`) by averaging predicted probabilities, which is a natural next step beyond this notebook.

---

## Future Work

- **Ensembling:** average softmax probabilities from this model with a second, diverse model (different pretraining corpus/tokenizer) to reduce variance and typically gain 1-3% accuracy.
- **Longer context:** head+tail truncation (keeping both the start and end of long documents) instead of head-only truncation, to recover information lost from truncated document tails.
- **Targeted class balancing:** light, capped class-weighting (not full "balanced" weighting, which destabilized an earlier run) specifically for the classes flagged as "never predicted" in Section 9.
